#### Criação da tabela com schema definido

In [0]:
create table if not exists 
  workspace.pj_sales.tb_fact_sale_gold (
     invoice_id         string
    ,part_quantity      int
    ,part_unit_price    double
    ,part_unit_cost     double
    ,_sk_date           bigint
    ,_sk_customer       int
    ,_sk_car            int
    ,_sk_store          int
    ,_sk_seller         int
  )

#### Criação da view temporária deduplicada por data de updating, de ingestion e de sale; adicionando _sks das dimensões

In [0]:
create or replace temp view vw_fact_sale_sk as
  select
     tss.invoice_id
    ,tss.part_quantity
    ,tss.part_unit_price
    ,tss.part_unit_cost
    ,coalesce(td._sk_date,     -1)   as _sk_date
    ,coalesce(tc._sk_customer, -1)   as _sk_customer
    ,coalesce(tcar._sk_car,    -1)   as _sk_car
    ,coalesce(tst._sk_store,   -1)   as _sk_store
    ,coalesce(ts._sk_seller,   -1)   as _sk_seller
    ,tss.sale_date
    ,tss.ingestion_timestamp
    ,tss.updating_timestamp
  from      workspace.pj_sales.tb_sales_silver tss
  left join workspace.pj_sales.tb_dim_date_gold td
    on      tss.sale_date = td.date
  left join workspace.pj_sales.tb_dim_seller_gold ts
    on      tss.seller_id = ts.seller_id
    and     (tss.sale_date < ts.valid_to
    or      ts.valid_to is null)
  left join workspace.pj_sales.tb_dim_car_gold tcar
    on      tss.car_brand = tcar.car_brand
    and     tss.car_model = tcar.car_model
    and     tss.car_part = tcar.car_part
    and     (tss.sale_date < tcar.valid_to 
    or      tcar.valid_to is null) -- Trata e pega a versão da dimensão referente a data da venda
  left join workspace.pj_sales.tb_dim_store_gold tst
    on      tss.store_id = tst.store_id
    and     (tss.sale_date < tst.valid_to 
    or      tst.valid_to is null)
  left join workspace.pj_sales.tb_dim_customer_gold tc
    on      tss.customer_id = tc.customer_id
    and     (tss.sale_date < tc.valid_to 
    or      tc.valid_to is null)
  where tss.invoice_id is not null --> Filtro aplicado apenas na gold

In [0]:
create or replace temp view vw_fact_sale as(
  with dedup as (
    select
       invoice_id
      ,part_quantity
      ,part_unit_price
      ,part_unit_cost
      ,_sk_date
      ,_sk_customer
      ,_sk_car
      ,_sk_store
      ,_sk_seller
      ,row_number() over (
        partition by
           invoice_id
        order by 
           sale_date           desc
          ,updating_timestamp  desc
          ,ingestion_timestamp desc
      ) as order_by_date
    from vw_fact_sale_sk
  )
  select
     invoice_id
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    ,_sk_date
    ,_sk_customer
    ,_sk_car
    ,_sk_store
    ,_sk_seller
  from dedup
  where order_by_date = 1
)

In [0]:
merge into workspace.pj_sales.tb_fact_sale_gold tg
using vw_fact_sale src
on  tg.invoice_id = src.invoice_id
when matched then
  update 
    set *
when not matched then
  insert *

In [0]:
select 
  invoice_id,
  _sk_car,
  count(*) as qtd
from workspace.pj_sales.tb_fact_sale_gold
group by invoice_id, _sk_car
having count(*) > 1 -- Verificação de grain (1 venda -> 1 produto)

In [0]:
drop view vw_fact_sale_sk;
drop view vw_fact_sale;